Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(37)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 24
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 5)
Dimensiones de Y: (52381, 1)


In [15]:
print(datosY[0])

[-1.59375341]


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 5)
Las dimensiones de testX son:  (10529, 12, 5)
Las dimensiones de valX son:  (5186, 12, 5)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

1146/1146 - 57s - 49ms/step - ia: 0.5085 - loss: 0.6067 - mae: 0.6334 - rmse: 0.7696 - smape: 1.1461 - val_ia: 0.2861 - val_loss: 0.8767 - val_mae: 0.7783 - val_rmse: 0.8430 - val_smape: 1.2252

Epoch 2/128                                           

1146/1146 - 27s - 24ms/step - ia: 0.6525 - loss: 0.3908 - mae: 0.5014 - rmse: 0.6201 - smape: 0.8963 - val_ia: 0.2893 - val_loss: 0.9018 - val_mae: 0.7791 - val_rmse: 0.8448 - val_smape: 1.1804

Epoch 3/128                                           

1146/1146 - 46s - 40ms/step - ia: 0.6887 - loss: 0.3367 - mae: 0.4613 - rmse: 0.5756 - smape: 0.8325 - val_ia: 0.2995 - val_loss: 0.7411 - val_mae: 0.7119 - val_rmse: 0.7755 - val_smape: 1.1236

Epoch 4/128                                           

1146/1146 - 29s - 25ms/step - ia: 0.7049 - loss: 0.3084 - mae: 0.4394 - rmse: 0.5505 - smape: 0.8094 - val_ia: 0.3057 - val_loss: 0.6868 - val_mae: 0.6767 - val_rmse: 0.7454 - val_smape: 1.06

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                          

144/144 - 52s - 362ms/step - ia: 0.6397 - loss: 0.3715 - mae: 0.4688 - rmse: 0.5830 - smape: 0.9209 - val_ia: 0.6020 - val_loss: 0.5068 - val_mae: 0.5843 - val_rmse: 0.6800 - val_smape: 1.0461

Epoch 2/16                                                                          

144/144 - 37s - 259ms/step - ia: 0.8207 - loss: 0.1425 - mae: 0.2841 - rmse: 0.3755 - smape: 0.6389 - val_ia: 0.6889 - val_loss: 0.2994 - val_mae: 0.4360 - val_rmse: 0.5310 - val_smape: 0.9005

Epoch 3/16                                                                          

144/144 - 17s - 116ms/step - ia: 0.8548 - loss: 0.0986 - mae: 0.2309 - rmse: 0.3129 - smape: 0.5761 - val_ia: 0.7384 - val_loss: 0.2117 - val_mae: 0.3572 - val_rmse: 0.4407 - val_smape: 0.7567

Epoch 4/16                                                                          

144/144 - 20s - 137ms/step - ia: 0.8678 - loss: 0.0846 - mae: 0.2114 - rms

...

4584/4584 - 74s - 16ms/step - ia: 0.7781 - loss: 0.1734 - mae: 0.3101 - rmse: 0.3869 - smape: 0.6692 - val_ia: 0.2767 - val_loss: 0.1899 - val_mae: 0.3355 - val_rmse: 0.3501 - val_smape: 0.6809

Epoch 2/8                                                                            

4584/4584 - 63s - 14ms/step - ia: 0.8372 - loss: 0.0964 - mae: 0.2296 - rmse: 0.2925 - smape: 0.5445 - val_ia: 0.2802 - val_loss: 0.1460 - val_mae: 0.2977 - val_rmse: 0.3157 - val_smape: 0.6386

Epoch 3/8                                                                            

4584/4584 - 61s - 13ms/step - ia: 0.8548 - loss: 0.0787 - mae: 0.2067 - rmse: 0.2639 - smape: 0.4954 - val_ia: 0.2333 - val_loss: 0.2309 - val_mae: 0.3770 - val_rmse: 0.3972 - val_smape: 0.7411

Epoch 4/8                                                                            

4584/4584 - 60s - 13ms/step - ia: 0.8636 - loss: 0.0690 - mae: 0.1933 - rmse: 0.2471 - smape: 0.4741 - val_ia: 0.2477 - val_loss: 0.2055 - val_mae: 0.3490

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



573/573 - 11s - 20ms/step - ia: 0.3611 - loss: 0.9885 - mae: 0.8382 - rmse: 0.9921 - smape: 1.3854 - val_ia: 0.2812 - val_loss: 0.8013 - val_mae: 0.7500 - val_rmse: 0.8600 - val_smape: 1.3459

Epoch 2/16                                                                           

573/573 - 5s - 9ms/step - ia: 0.3499 - loss: 0.9321 - mae: 0.8151 - rmse: 0.9634 - smape: 1.3968 - val_ia: 0.2872 - val_loss: 0.7806 - val_mae: 0.7384 - val_rmse: 0.8468 - val_smape: 1.3560

Epoch 3/16                                                                           

573/573 - 5s - 9ms/step - ia: 0.3455 - loss: 0.8876 - mae: 0.7936 - rmse: 0.9402 - smape: 1.4023 - val_ia: 0.2934 - val_loss: 0.7678 - val_mae: 0.7304 - val_rmse: 0.8384 - val_smape: 1.3640

Epoch 4/16                                                                           

573/573 - 5s - 9ms/step - ia: 0.3366 - loss: 0.8622 - mae: 0.7821 - rmse: 0.9268 - smape: 1.4097 - val_ia: 0.2990 - val_loss: 0.7594 - val_mae: 0.7247 - val_rmse: 0

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



287/287 - 12s - 42ms/step - ia: 0.3033 - loss: 1.5565 - mae: 1.0118 - rmse: 1.2440 - smape: 1.4331 - val_ia: 0.2534 - val_loss: 0.8918 - val_mae: 0.7638 - val_rmse: 0.9344 - val_smape: 1.2844

Epoch 2/32                                                                           

287/287 - 4s - 15ms/step - ia: 0.2635 - loss: 1.2923 - mae: 0.9303 - rmse: 1.1351 - smape: 1.4921 - val_ia: 0.2463 - val_loss: 0.8350 - val_mae: 0.7529 - val_rmse: 0.9079 - val_smape: 1.4954

Epoch 3/32                                                                           

287/287 - 4s - 15ms/step - ia: 0.2463 - loss: 1.2199 - mae: 0.9073 - rmse: 1.1026 - smape: 1.5094 - val_ia: 0.2283 - val_loss: 0.8426 - val_mae: 0.7619 - val_rmse: 0.9119 - val_smape: 1.7225

Epoch 4/32                                                                           

287/287 - 4s - 15ms/step - ia: 0.2434 - loss: 1.1860 - mae: 0.8948 - rmse: 1.0875 - smape: 1.5107 - val_ia: 0.2199 - val_loss: 0.8507 - val_mae: 0.7669 - val_rmse

In [22]:
print(best)

{'activation': 1, 'batch': 1, 'dropout': 0.0, 'epochs': 2, 'layers': 4.0, 'learning_rate': 0.0007611975499647884, 'units': 0}
